# 不動産相場チェッカー

住所を入力するだけで、国土交通省の実際の取引データから**坪単価・一種単価**を自動分析します。

## 使い方

1. **APIキーを取得**: [不動産情報ライブラリ API利用申請](https://www.reinfolib.mlit.go.jp/api/request/) から無料で申請できます
2. 下のセルを順番に実行（「ランタイム」→「すべてのセルを実行」でもOK）
3. APIキーの入力欄が表示されるので、キーを貼り付けてEnter

## 何がわかるの？

- **坪単価**（つぼたんか）: 土地1坪（約3.3m2）あたりの価格
- **一種単価**（いっしゅたんか）: 容積率で割った坪単価。条件が違う土地を公平に比較できる
- **建物残価推定**: 建物付き取引から建物の価値を差し引いて、土地だけの価格を逆算した参考値

## セキュリティについて

- APIキーは `getpass` で入力されるため、**ノートブックを保存してもキーは残りません**
- コードは国交省API (`reinfolib.mlit.go.jp`) と国土地理院 (`gsi.go.jp`) にのみ通信します
- 全通信はHTTPS暗号化されています
- **実行後にこのノートブックを他人と共有すると、入力した住所や分析結果が見える可能性があります**。共有する場合は「編集」→「出力を全て消去」してからにしてください

In [ ]:
#@title 1. 設定 { display-mode: "form" }
import getpass

#@markdown ## 住所を入力してください
#@markdown 番地まで入れなくてOK（例: 渋谷区恵比寿、蒲田5丁目、練馬区中村）
address = "\u6E0B\u8C37\u533A\u6075\u6BD4\u5BFF"  #@param {type:"string"}
#@markdown ---
#@markdown 何年分のデータを取得するか（デフォルト3年）
years_back = 3  #@param {type:"slider", min:1, max:5, step:1}

#@markdown ---
#@markdown ## APIキーを入力（下の入力欄にキーを貼り付けてEnter）
api_key = getpass.getpass("APIキーを入力してください: ")
if api_key:
    print("APIキーを受け付けました（表示されません）")
else:
    print("APIキーが入力されていません")

In [ ]:
#@title 2. 分析エンジン（自動実行されます。このセルは編集不要） { display-mode: "form" }

import datetime
import re
import statistics
import time

import requests

# ─── 定数 ─────────────────────────────────────────────────

REINFOLIB_API_URL = "https://www.reinfolib.mlit.go.jp/ex-api/external"
GSI_GEOCODER_URL = "https://msearch.gsi.go.jp/address-search/AddressSearch"
GSI_REVERSE_GEOCODER_URL = (
    "https://mreversegeocoder.gsi.go.jp/reverse-geocoder/LonLatToAddress"
)

TSUBO_PER_SQM = 3.30579

PREFECTURE_CODES = {
    "01": "北海道", "02": "青森県", "03": "岩手県", "04": "宮城県",
    "05": "秋田県", "06": "山形県", "07": "福島県", "08": "茨城県",
    "09": "栃木県", "10": "群馬県", "11": "埼玉県", "12": "千葉県",
    "13": "東京都", "14": "神奈川県", "15": "新潟県", "16": "富山県",
    "17": "石川県", "18": "福井県", "19": "山梨県", "20": "長野県",
    "21": "岐阜県", "22": "静岡県", "23": "愛知県", "24": "三重県",
    "25": "滋賀県", "26": "京都府", "27": "大阪府", "28": "兵庫県",
    "29": "奈良県", "30": "和歌山県", "31": "鳥取県", "32": "島根県",
    "33": "岡山県", "34": "広島県", "35": "山口県", "36": "徳島県",
    "37": "香川県", "38": "愛媛県", "39": "高知県", "40": "福岡県",
    "41": "佐賀県", "42": "長崎県", "43": "熊本県", "44": "大分県",
    "45": "宮崎県", "46": "鹿児島県", "47": "沖縄県",
}

BUILDING_SPECS = {
    "SRC": {"useful_life": 47, "unit_cost": 335_000},
    "RC":  {"useful_life": 47, "unit_cost": 334_000},
    "LS":  {"useful_life": 27, "unit_cost": 325_000},
    "S":   {"useful_life": 34, "unit_cost": 325_000},
    "W":   {"useful_life": 22, "unit_cost": 215_000},
}


# ─── 住所 → 座標変換 ──────────────────────────────────────

def geocode(address):
    resp = requests.get(GSI_GEOCODER_URL, params={"q": address}, timeout=10, allow_redirects=False)
    resp.raise_for_status()
    data = resp.json()
    if not data:
        raise ValueError(f"住所が見つかりません: {address}")
    lon, lat = data[0]["geometry"]["coordinates"]
    return lat, lon


def reverse_geocode(lat, lon):
    resp = requests.get(
        GSI_REVERSE_GEOCODER_URL, params={"lat": lat, "lon": lon}, timeout=10, allow_redirects=False
    )
    resp.raise_for_status()
    result = resp.json()
    return result["results"]["muniCd"], result["results"]["lv01Nm"]


# ─── 住所解析ヘルパー ─────────────────────────────────────

def get_pref_name(muni_cd):
    return PREFECTURE_CODES.get(muni_cd[:2], "")

def get_district_name(lv01_nm):
    return re.sub(r"([一二三四五六七八九十百千万]+|[0-9]+)丁目$", "", lv01_nm)

def get_city_name(key, muni_cd):
    pref_cd = muni_cd[:2]
    resp = requests.get(
        f"{REINFOLIB_API_URL}/XIT002",
        params={"area": pref_cd},
        headers={"Ocp-Apim-Subscription-Key": key},
        timeout=10,
        allow_redirects=False,
    )
    resp.raise_for_status()
    data = resp.json()
    match = next((item for item in data.get("data", []) if item["id"] == muni_cd), None)
    return match["name"] if match else None


# ─── 取引データ取得 ────────────────────────────────────────

def fetch_transactions(key, muni_cd, years):
    all_data = []
    for i, year in enumerate(years):
        if i > 0:
            time.sleep(1)
        resp = requests.get(
            f"{REINFOLIB_API_URL}/XIT001",
            params={"year": year, "city": muni_cd},
            headers={"Ocp-Apim-Subscription-Key": key},
            timeout=30,
            allow_redirects=False,
        )
        if resp.status_code == 404:
            continue
        resp.raise_for_status()
        result = resp.json()
        if result and "data" in result:
            all_data.extend(result["data"])
    return all_data


# ─── データフィルタリング ─────────────────────────────────

def filter_by_district(data, prefecture, municipality, district_name):
    filtered = []
    for item in data:
        if item.get("Prefecture") != prefecture:
            continue
        item_muni = item.get("Municipality", "")
        if municipality:
            muni_match = (
                item_muni == municipality
                or (municipality.endswith("区") and item_muni.endswith(municipality))
            )
            if not muni_match:
                continue
        if item.get("DistrictName") != district_name:
            continue
        filtered.append(item)
    return filtered


# ─── 和暦変換 ─────────────────────────────────────────────

def convert_japanese_year(year_str):
    if not year_str:
        return None
    year_str = year_str.replace("元年", "1年")
    patterns = [
        (r"令和(\d+)年", lambda m: 2018 + int(m.group(1))),
        (r"平成(\d+)年", lambda m: 1988 + int(m.group(1))),
        (r"昭和(\d+)年", lambda m: 1925 + int(m.group(1))),
    ]
    for pattern, converter in patterns:
        m = re.search(pattern, year_str)
        if m:
            return converter(m)
    m = re.search(r"(\d{4})年", year_str)
    if m:
        return int(m.group(1))
    return None


# ─── 建物残価推定 ─────────────────────────────────────────

def _classify_structure(structure):
    if "ＳＲＣ" in structure:
        return BUILDING_SPECS["SRC"]
    if "ＲＣ" in structure:
        return BUILDING_SPECS["RC"]
    if "軽量鉄骨" in structure:
        return BUILDING_SPECS["LS"]
    if "鉄骨" in structure:
        return BUILDING_SPECS["S"]
    return BUILDING_SPECS["W"]


def estimate_building_residual(item):
    trade_price = int(item.get("TradePrice", 0) or 0)
    total_floor_area = item.get("TotalFloorArea")
    building_year = item.get("BuildingYear")
    structure = item.get("Structure", "")
    if not all([trade_price, total_floor_area, building_year]):
        return None
    try:
        floor_area = float(total_floor_area)
    except (ValueError, TypeError):
        return None
    current_year = datetime.date.today().year
    built_year = convert_japanese_year(building_year)
    if not built_year:
        return None
    age = current_year - built_year
    specs = _classify_structure(structure)
    remaining_ratio = max(0, 1 - age / specs["useful_life"])
    building_value = specs["unit_cost"] * floor_area * remaining_ratio
    estimated_land_price = trade_price - building_value
    if estimated_land_price <= 0:
        return None
    return {
        "estimated_land_price": int(estimated_land_price),
        "building_value": int(building_value),
        "remaining_ratio": remaining_ratio,
    }


# ─── 坪単価・一種単価算出 ─────────────────────────────────

def calc_tsubo_stats(items):
    results = []
    for item in items:
        if item.get("Type") != "宅地(土地のみ)":
            continue
        trade_price = int(item.get("TradePrice", 0) or 0)
        if trade_price <= 100_000:
            continue
        area_m2 = float(item.get("Area", 0) or 0)
        if area_m2 <= 0:
            continue
        tsubo_price = trade_price / area_m2 * TSUBO_PER_SQM
        floor_area_ratio = float(item.get("FloorAreaRatio") or 0)
        ikishu_price = (
            tsubo_price / (floor_area_ratio / 100) if floor_area_ratio > 0 else None
        )
        results.append({
            "tsubo_price": tsubo_price, "ikishu_price": ikishu_price,
            "trade_price": trade_price, "area_m2": area_m2,
            "floor_area_ratio": floor_area_ratio,
            "use_district": item.get("UseDistrict", ""),
            "period": item.get("Period", ""),
        })
    return results


def calc_building_land_stats(items):
    results = []
    for item in items:
        if item.get("Type") != "宅地(土地と建物)":
            continue
        area_m2 = float(item.get("Area", 0) or 0)
        if area_m2 <= 0:
            continue
        estimation = estimate_building_residual(item)
        if estimation is None:
            continue
        tsubo_price = estimation["estimated_land_price"] / area_m2 * TSUBO_PER_SQM
        floor_area_ratio = float(item.get("FloorAreaRatio") or 0)
        ikishu_price = (
            tsubo_price / (floor_area_ratio / 100) if floor_area_ratio > 0 else None
        )
        results.append({
            "tsubo_price": tsubo_price, "ikishu_price": ikishu_price,
            "estimated_land_price": estimation["estimated_land_price"],
            "building_value": estimation["building_value"],
            "area_m2": area_m2, "floor_area_ratio": floor_area_ratio,
            "use_district": item.get("UseDistrict", ""),
            "period": item.get("Period", ""),
        })
    return results


# ─── 異常値除外（2パス目）──────────────────────────────────

def remove_outliers(results):
    tsubo_prices = [r["tsubo_price"] for r in results if r["tsubo_price"] is not None]
    if len(tsubo_prices) < 3:
        return results
    median = statistics.median(tsubo_prices)
    return [r for r in results if r["tsubo_price"] is None or r["tsubo_price"] >= median / 5]


# ─── 統計サマリー ─────────────────────────────────────────

def summarize_stats(results):
    tsubo_prices = [r["tsubo_price"] for r in results]
    ikishu_prices = [r["ikishu_price"] for r in results if r["ikishu_price"] is not None]
    summary = {
        "count": len(tsubo_prices),
        "tsubo_min": min(tsubo_prices) if tsubo_prices else None,
        "tsubo_max": max(tsubo_prices) if tsubo_prices else None,
        "tsubo_median": statistics.median(tsubo_prices) if tsubo_prices else None,
    }
    if ikishu_prices:
        summary["ikishu_min"] = min(ikishu_prices)
        summary["ikishu_max"] = max(ikishu_prices)
        summary["ikishu_median"] = statistics.median(ikishu_prices)
    return summary


def summarize_by_use_district(results):
    groups = {}
    for r in results:
        key = r.get("use_district") or "不明"
        groups.setdefault(key, []).append(r)
    return {district: summarize_stats(items) for district, items in groups.items()}


print("分析エンジンの読み込み完了")

In [ ]:
#@title 3. 分析を実行！ { display-mode: "form" }
#@markdown ▶ ボタンを押すと、APIからデータを取得して分析します（10〜30秒かかります）

import pandas as pd
from IPython.display import display, Markdown, HTML

# --- 入力チェック ---
if not api_key or not api_key.strip():
    print("APIキーが入力されていません。セル1でAPIキーを入力してください。")
    raise SystemExit
if not address or not address.strip():
    print("住所が入力されていません。セル1で住所を入力してください。")
    raise SystemExit

# --- 年の計算 ---
current_year = datetime.date.today().year
years = [current_year - i for i in range(years_back)]

print(f"分析開始: {address}")
print(f"対象年: {', '.join(map(str, years))}")
print()

# --- Step 1: 住所 → 座標 ---
print("Step 1/5: 住所から座標を取得中...")
try:
    lat, lon = geocode(address)
    print(f"  座標: ({lat:.6f}, {lon:.6f})")
except Exception as e:
    print(f"  エラー: {e}")
    raise SystemExit

# --- Step 2: 座標 → 市区町村コード ---
print("Step 2/5: 市区町村を特定中...")
try:
    muni_cd, lv01_nm = reverse_geocode(lat, lon)
    prefecture = get_pref_name(muni_cd)
    municipality = get_city_name(api_key, muni_cd)
    district_name = get_district_name(lv01_nm)
    full_addr = f"{prefecture}{municipality}{district_name}"
    print(f"  {full_addr} (コード: {muni_cd})")
except Exception as e:
    print(f"  エラー: {e}")
    print("  APIキーが正しいか確認してください。")
    raise SystemExit

# --- Step 3: 取引データ取得 ---
print("Step 3/5: 取引データを取得中（API呼び出し間に1秒待機）...")
try:
    raw_data = fetch_transactions(api_key, muni_cd, years)
    print(f"  取得件数: {len(raw_data)}件")
except Exception as e:
    print(f"  エラー: {e}")
    raise SystemExit

# --- Step 4: フィルタリング ---
print("Step 4/5: 対象地区のデータを抽出中...")
filtered = filter_by_district(raw_data, prefecture, municipality, district_name)
print(f"  対象地区の取引: {len(filtered)}件")

if len(filtered) == 0:
    print()
    print(f"「{full_addr}」の取引データが見つかりませんでした。")
    print("住所の表記を変えて試してみてください（例: 区名を追加、丁目を外すなど）")
    raise SystemExit

# --- Step 5: 分析 ---
print("Step 5/5: 坪単価・建物残価を計算中...")

# 土地のみ取引
land_only = calc_tsubo_stats(filtered)
land_only = remove_outliers(land_only)
land_stats = summarize_by_use_district(land_only)
land_overall = summarize_stats(land_only)

# 建物込み取引
building_land = calc_building_land_stats(filtered)
building_land = remove_outliers(building_land)
building_stats = summarize_by_use_district(building_land)
building_overall = summarize_stats(building_land)

print("  完了!")
print()

# === 結果表示 ===

def fmt(value):
    """円を万円表記に"""
    if value is None:
        return "-"
    return f"{value / 10000:.1f}"

display(Markdown(f"# 不動産相場分析レポート: {address}"))
display(Markdown(f"**住所**: {full_addr} / **座標**: ({lat:.6f}, {lon:.6f}) / **対象年**: {', '.join(map(str, years))}"))
display(Markdown(f"全取引 {len(raw_data)}件 → 対象地区 {len(filtered)}件"))

# --- 土地のみ取引テーブル ---
display(Markdown(f"## 土地のみ取引（{len(land_only)}件）"))
if len(land_only) == 0:
    display(Markdown("該当データなし"))
else:
    if len(land_only) < 3:
        display(Markdown("**注意: 3件未満のためデータ不足。暫定参考値です。**"))
    rows = []
    for district, stats in land_stats.items():
        rows.append({
            "用途地域": district,
            "件数": stats["count"],
            "坪単価 最小(万円)": fmt(stats["tsubo_min"]),
            "坪単価 中央値(万円)": fmt(stats["tsubo_median"]),
            "坪単価 最大(万円)": fmt(stats["tsubo_max"]),
            "一種単価 中央値(万円)": fmt(stats.get("ikishu_median")),
        })
    display(pd.DataFrame(rows))
    display(Markdown(f"**全体 坪単価 中央値: {fmt(land_overall['tsubo_median'])}万円**"))

# --- 建物込み取引テーブル ---
display(Markdown(f"## 建物込み取引からの推定土地坪単価（参考）（{len(building_land)}件）"))
if len(building_land) == 0:
    display(Markdown("該当データなし"))
else:
    rows = []
    for district, stats in building_stats.items():
        rows.append({
            "用途地域": district,
            "件数": stats["count"],
            "推定坪単価 最小(万円)": fmt(stats["tsubo_min"]),
            "推定坪単価 中央値(万円)": fmt(stats["tsubo_median"]),
            "推定坪単価 最大(万円)": fmt(stats["tsubo_max"]),
        })
    display(pd.DataFrame(rows))
    display(Markdown(f"**全体 推定坪単価 中央値: {fmt(building_overall['tsubo_median'])}万円**"))

# --- 免責表示 ---
display(Markdown("---"))
display(Markdown(
    "> このサービスは、国土交通省不動産情報ライブラリのAPI機能を使用していますが、"
    "提供情報の最新性、正確性、完全性等が保証されたものではありません。\n"
    "> 建物込み取引の推定土地価格は、法定耐用年数に基づく簡易推定であり、"
    "リフォーム・個別品質・設備は未考慮です。参考値としてご利用ください。"
))